In [9]:
import pandas as pd
import requests

from bs4 import BeautifulSoup
from geopy.geocoders import Nominatim

from google.cloud import bigquery

from datetime import datetime, timezone

import json
import time

from google import genai
from google.genai import types
import json

In [2]:
PROJECT_ID = "pacey32-agency"

TEAM_SQL = """
SELECT
    org.id,
    org.fullName,
    org.tricode,
    org.venue,
    org.venueLocation,
    city.state_province,
    city.country,
    city.latitude AS city_latitude,
    city.longitude AS city_longitude
FROM `pacey32-agency.Team.OrganizationDetail` AS org
LEFT JOIN `pacey32-agency.City.CityReference` AS city
    ON org.venueLocation = city.city_name
ORDER BY org.fullName
"""

In [3]:
client = bigquery.Client(project=PROJECT_ID)

teams = client.query(
    TEAM_SQL
).to_dataframe(
    create_bqstorage_client=False
)

In [4]:
#teams.head(32)

In [5]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="nhl_map_builder")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
)

arena_results = []

for _, row in teams.iterrows():

    query = (
        f"{row.venue}, "
        f"{row.venueLocation}, "
        f"{row.state_province}, "
        f"{row.country}"
    )

    location = geocode(query)

    arena_results.append({
        "fullName": row.fullName,
        "arena_name": row.venue,
        "query": query,
        "arena_latitude": location.latitude if location else None,
        "arena_longitude": location.longitude if location else None,
        "matched_address": location.address if location else None,
    })

arena_df = pd.DataFrame(arena_results)

In [7]:
#display(arena_df)

In [ ]:
client = genai.Client(api_key="ABC123")

SYSTEM_PROMPT = """
Return JSON only.

Find the primary practice/training facility currently used by the NHL team.

Return:

{
  "facility_name": "...",
  "address": "...",
  "city": "...",
  "state_province": "...",
  "country": "..."
}

Do not include markdown.
Use the primary day-to-day practice facility.
"""

def get_practice_facility(team):

    response = client.models.generate_content(
        model="models/gemini-3.1-flash-lite",
        contents=f"What is the primary practice facility for the {team}?",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0,
            response_mime_type="application/json",
        ),
    )

    return json.loads(response.text)

In [13]:
facility = get_practice_facility("Washington Capitals")
facility

{'facility_name': 'MedStar Capitals Iceplex',
 'address': '627 N Glebe Rd',
 'city': 'Arlington',
 'state_province': 'Virginia',
 'country': 'United States'}

In [14]:
practice_results = []

for _, row in teams.iterrows():

    print(f"Getting {row.fullName}...")

    try:
        facility = get_practice_facility(row.fullName)

        practice_results.append({
            "id": row.id,
            "fullName": row.fullName,
            "tricode": row.tricode,
            "facility_name": facility.get("facility_name"),
            "address": facility.get("address"),
            "city": facility.get("city"),
            "state_province": facility.get("state_province"),
            "country": facility.get("country"),
        })

    except Exception as e:

        print(f"Failed: {row.fullName} - {e}")

        practice_results.append({
            "id": row.id,
            "fullName": row.fullName,
            "tricode": row.tricode,
            "facility_name": None,
            "address": None,
            "city": None,
            "state_province": None,
            "country": None,
        })

    # Stay comfortably under the Gemini rate limit
    time.sleep(13)

practice_df = pd.DataFrame(practice_results)

display(practice_df)

Getting Anaheim Ducks...
Getting Boston Bruins...
Getting Buffalo Sabres...
Getting Calgary Flames...
Getting Carolina Hurricanes...
Getting Chicago Blackhawks...
Getting Colorado Avalanche...
Getting Columbus Blue Jackets...
Getting Dallas Stars...
Getting Detroit Red Wings...
Getting Edmonton Oilers...
Getting Florida Panthers...
Getting Los Angeles Kings...
Getting Minnesota Wild...
Getting Montréal Canadiens...
Getting Nashville Predators...
Getting New Jersey Devils...
Getting New York Islanders...
Getting New York Rangers...
Getting Ottawa Senators...
Getting Philadelphia Flyers...
Getting Pittsburgh Penguins...
Getting San Jose Sharks...
Getting Seattle Kraken...
Getting St. Louis Blues...
Getting Tampa Bay Lightning...
Getting Toronto Maple Leafs...
Getting Utah Mammoth...
Getting Vancouver Canucks...
Getting Vegas Golden Knights...
Getting Washington Capitals...
Getting Winnipeg Jets...


,id,fullName,tricode,facility_name,address,city,state_province,country
0,24,Anaheim Ducks,ANA,Great Park Ice & FivePoint Arena,888 Ridge Valley,Irvine,California,United States
1,6,Boston Bruins,BOS,Warrior Ice Arena,90 Guest St,Boston,Massachusetts,USA
2,7,Buffalo Sabres,BUF,LECOM Harborcenter,100 Washington St,Buffalo,New York,United States
3,20,Calgary Flames,CGY,WinSport Event Centre,151 Canada Olympic Rd SW,Calgary,Alberta,Canada
4,12,Carolina Hurricanes,CAR,Lenovo Center,1400 Edwards Mill Rd,Raleigh,North Carolina,USA
5,16,Chicago Blackhawks,CHI,Fifth Third Arena,1801 W Jackson Blvd,Chicago,Illinois,United States
6,21,Colorado Avalanche,COL,Family Sports Center,6901 S Peoria St,Centennial,Colorado,United States
7,29,Columbus Blue Jackets,CBJ,OhioHealth Ice Haus,200 W Nationwide Blvd,Columbus,Ohio,United States
8,25,Dallas Stars,DAL,Children's Health StarCenter Frisco,2601 Avenue of the Stars,Frisco,Texas,United States
9,17,Detroit Red Wings,DET,Little Caesars Arena,2645 Woodward Ave,Detroit,Michigan,United States


In [15]:
practice_df.isna().sum()

id                0
fullName          0
tricode           0
facility_name     0
address           0
city              0
state_province    0
country           0
dtype: int64

In [16]:
practice_df.isna().sum()

id                0
fullName          0
tricode           0
facility_name     0
address           0
city              0
state_province    0
country           0
dtype: int64

In [17]:
practice_geocoded = []

for _, row in practice_df.iterrows():

    query = ", ".join(
        x for x in [
            row.facility_name,
            row.address,
            row.city,
            row.state_province,
            row.country,
        ]
        if pd.notna(x)
    )

    location = geocode(query)

    practice_geocoded.append({
        "id": row.id,
        "fullName": row.fullName,
        "facility_name": row.facility_name,
        "practice_latitude": location.latitude if location else None,
        "practice_longitude": location.longitude if location else None,
        "matched_address": location.address if location else None,
    })

practice_geo_df = pd.DataFrame(practice_geocoded)

display(practice_geo_df)

,id,fullName,facility_name,practice_latitude,practice_longitude,matched_address
0,24,Anaheim Ducks,Great Park Ice & FivePoint Arena,33.677682,-117.745157,"Great Park Ice & FivePoint Arena, 888, Ridge V..."
1,6,Boston Bruins,Warrior Ice Arena,42.357477,-71.144052,"Warrior Ice Arena, 90, Guest Street, Boston La..."
2,7,Buffalo Sabres,LECOM Harborcenter,42.876355,-78.876813,"LECOM Harborcenter, 100, Washington Street, Ca..."
3,20,Calgary Flames,WinSport Event Centre,NaN,NaN,NaN
4,12,Carolina Hurricanes,Lenovo Center,35.803398,-78.721917,"Lenovo Center, 1400, Hurricanes Highway, Wade,..."
5,16,Chicago Blackhawks,Fifth Third Arena,41.877048,-87.673596,"Fifth Third Arena, 1801, West Jackson Boulevar..."
6,21,Colorado Avalanche,Family Sports Center,39.591968,-104.849030,"Family Sports Center, 6901, South Peoria Stree..."
7,29,Columbus Blue Jackets,OhioHealth Ice Haus,NaN,NaN,NaN
8,25,Dallas Stars,Children's Health StarCenter Frisco,NaN,NaN,NaN
9,17,Detroit Red Wings,Little Caesars Arena,42.340977,-83.054954,"Little Caesars Arena, 2645, Woodward Avenue, C..."


In [18]:
team_locations = (
    arena_df
    .merge(
        practice_geo_df,
        on=["id", "fullName"],
        how="left",
    )
)

display(team_locations)

KeyError: 'id'